In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/processed/vendas_limpas_loja_moda.csv",
                 encoding="utf-8-sig", parse_dates=["data_venda"])

print(f"Shape: {df.shape}")
print(f"Período: {df['data_venda'].min().date()} a {df['data_venda'].max().date()}")
df.head()

ModuleNotFoundError: No module named 'pandas'

In [ ]:
total_faturamento = df["valor_total"].sum()
total_vendas = len(df)
ticket_medio = df["valor_total"].mean()
total_clientes = df["nome_cliente"].nunique()
total_produtos = df["produto"].nunique()
media_itens_por_venda = df["quantidade"].mean()

print("=" * 45)
print("        RESUMO GERAL — ANO 2023")
print("=" * 45)
print(f"  Faturamento total:    R$ {total_faturamento:>10,.2f}")
print(f"  Total de vendas:      {total_vendas:>10}")
print(f"  Ticket médio:         R$ {ticket_medio:>10,.2f}")
print(f"  Clientes únicos:      {total_clientes:>10}")
print(f"  Produtos no mix:      {total_produtos:>10}")
print(f"  Média itens/venda:    {media_itens_por_venda:>10.1f}")
print("=" * 45)

        RESUMO GERAL — ANO 2023
  Faturamento total:    R$  77,927.18
  Total de vendas:             500
  Ticket médio:         R$     155.85
  Clientes únicos:              81
  Produtos no mix:              11
  Média itens/venda:           1.6


In [ ]:
ordem_meses = range(1, 13)
nomes_meses = {
    1: "Jan", 2: "Fev", 3: "Mar", 4: "Abr",
    5: "Mai", 6: "Jun", 7: "Jul", 8: "Ago",
    9: "Set", 10: "Out", 11: "Nov", 12: "Dez"
}

fat_mensal = (
    df.groupby("mes")["valor_total"]
    .sum()
    .reindex(ordem_meses, fill_value=0)
    .reset_index()
)
fat_mensal.columns = ["mes", "faturamento"]
fat_mensal["mes_nome"] = fat_mensal["mes"].map(nomes_meses)
fat_mensal["variacao_pct"] = fat_mensal["faturamento"].pct_change() * 100

print("Faturamento mensal:\n")
for _, row in fat_mensal.iterrows():
    barra = "█" * int(row["faturamento"] / fat_mensal["faturamento"].max() * 30)
    var = f"({row['variacao_pct']:+.1f}%)" if pd.notna(row["variacao_pct"]) else ""
    print(f"  {row['mes_nome']}  {barra:<30}  R$ {row['faturamento']:>8,.2f}  {var}")

melhor_mes = fat_mensal.loc[fat_mensal["faturamento"].idxmax()]
pior_mes   = fat_mensal.loc[fat_mensal["faturamento"].idxmin()]
print(f"\n  Melhor mês: {melhor_mes['mes_nome']} — R$ {melhor_mes['faturamento']:,.2f}")
print(f"  Pior mês:   {pior_mes['mes_nome']} — R$ {pior_mes['faturamento']:,.2f}")

Faturamento mensal:

  Jan  ████████████████                R$ 5,407.18  
  Fev  ██████████████████████          R$ 7,323.18  (+35.4%)
  Mar  ██████████████████████████████  R$ 9,899.30  (+35.2%)
  Abr  █████████████████               R$ 5,897.98  (-40.4%)
  Mai  ████████████████████████        R$ 8,023.00  (+36.0%)
  Jun  ███████████████                 R$ 5,074.05  (-36.8%)
  Jul  ████████████████████████        R$ 8,209.31  (+61.8%)
  Ago  ██████████████████████          R$ 7,329.54  (-10.7%)
  Set  ██████████████                  R$ 4,926.68  (-32.8%)
  Out  ██████████                      R$ 3,331.35  (-32.4%)
  Nov  ██████████████████              R$ 5,996.44  (+80.0%)
  Dez  ███████████████████             R$ 6,509.18  (+8.6%)

  Melhor mês: Mar — R$ 9,899.30
  Pior mês:   Out — R$ 3,331.35


In [ ]:
top_produtos = (
    df.groupby("produto")
    .agg(
        qtd_vendida=("quantidade", "sum"),
        faturamento=("valor_total", "sum"),
        preco_medio=("preco_unitario", "mean")
    )
    .sort_values("faturamento", ascending=False)
    .reset_index()
)
top_produtos["participacao_pct"] = (
    top_produtos["faturamento"] / top_produtos["faturamento"].sum() * 100
)

print("Top produtos por faturamento:\n")
print(f"  {'Produto':<25} {'Qtd':>6} {'Faturamento':>12} {'Part.':>7} {'Preço médio':>12}")
print(f"  {'-'*25} {'-'*6} {'-'*12} {'-'*7} {'-'*12}")
for _, row in top_produtos.iterrows():
    print(
        f"  {row['produto']:<25} "
        f"{row['qtd_vendida']:>6} "
        f"R$ {row['faturamento']:>9,.2f} "
        f"{row['participacao_pct']:>6.1f}% "
        f"R$ {row['preco_medio']:>8,.2f}"
    )

Top produtos por faturamento:

  Produto                      Qtd  Faturamento   Part.  Preço médio
  ------------------------- ------ ------------ ------- ------------
  Jaqueta Couro M               94 R$ 26,494.80   34.0% R$   284.63
  Calça Jeans 42                77 R$  9,514.83   12.2% R$   123.47
  Blusa de Frio G               92 R$  8,797.61   11.3% R$    95.28
  Vestido Floral P              81 R$  7,006.00    9.0% R$    85.81
  Tênis Casual 38               50 R$  6,325.28    8.1% R$   132.04
  Short Jeans 40                70 R$  5,351.92    6.9% R$    76.53
  Camiseta Básica M            107 R$  5,035.09    6.5% R$    47.14
  Sandália Rasteira 36          86 R$  4,919.25    6.3% R$    58.25
  Boné Aba Curva                69 R$  2,847.99    3.7% R$    41.77
  Meia Kit C/3                  43 R$  1,043.19    1.3% R$    24.13
  Meia Kit C/ 3                 25 R$    591.23    0.8% R$    23.73


In [ ]:
ticket_pagamento = (
    df.groupby("forma_pagamento")
    .agg(
        n_vendas=("id_venda", "count"),
        ticket_medio=("valor_total", "mean"),
        faturamento=("valor_total", "sum")
    )
    .sort_values("faturamento", ascending=False)
    .reset_index()
)

print("Ticket médio por forma de pagamento:\n")
print(f"  {'Pagamento':<22} {'Vendas':>7} {'Ticket médio':>14} {'Faturamento':>14}")
print(f"  {'-'*22} {'-'*7} {'-'*14} {'-'*14}")
for _, row in ticket_pagamento.iterrows():
    print(
        f"  {row['forma_pagamento']:<22} "
        f"{row['n_vendas']:>7} "
        f"R$ {row['ticket_medio']:>10,.2f} "
        f"R$ {row['faturamento']:>10,.2f}"
    )

Ticket médio por forma de pagamento:

  Pagamento               Vendas   Ticket médio    Faturamento
  ---------------------- ------- -------------- --------------
  Cartão de Crédito          135 R$     164.16 R$  22,161.65
  Pix                        139 R$     159.28 R$  22,140.36
  Cartão de Débito           119 R$     151.62 R$  18,043.37
  Dinheiro                   107 R$     145.62 R$  15,581.81


In [ ]:
top_clientes = (
    df.groupby("nome_cliente")
    .agg(
        n_compras=("id_venda", "count"),
        total_gasto=("valor_total", "sum"),
        ticket_medio=("valor_total", "mean"),
        ultima_compra=("data_venda", "max")
    )
    .sort_values("total_gasto", ascending=False)
    .head(10)
    .reset_index()
)

print("Top 10 clientes por valor gasto:\n")
print(f"  {'Cliente':<25} {'Compras':>8} {'Total gasto':>13} {'Ticket médio':>13} {'Última compra':>14}")
print(f"  {'-'*25} {'-'*8} {'-'*13} {'-'*13} {'-'*14}")
for _, row in top_clientes.iterrows():
    print(
        f"  {row['nome_cliente']:<25} "
        f"{row['n_compras']:>8} "
        f"R$ {row['total_gasto']:>9,.2f} "
        f"R$ {row['ticket_medio']:>9,.2f} "
        f"{str(row['ultima_compra'].date()):>14}"
    )

Top 10 clientes por valor gasto:

  Cliente                    Compras   Total gasto  Ticket médio  Última compra
  ------------------------- -------- ------------- ------------- --------------
  Mauricio Sampaio                 9 R$  2,578.20 R$    286.47     2023-09-30
  Alexandre Pires                 16 R$  2,498.59 R$    156.16     2023-12-13
  Felipe Ribeiro                   8 R$  2,390.04 R$    298.75     2023-11-29
  Simone Castro                    9 R$  2,276.02 R$    252.89     2023-12-16
  Valdinei Faria                   9 R$  2,079.34 R$    231.04     2023-12-25
  Rafael Mendes                    9 R$  1,957.48 R$    217.50     2023-11-07
  Lucia Ferreira                   9 R$  1,900.58 R$    211.18     2023-09-26
  Jairo Cavalcante                 9 R$  1,756.70 R$    195.19     2023-11-29
  Terezinha Barros                 8 R$  1,742.82 R$    217.85     2023-12-17
  Ailton Bastos                    8 R$  1,731.54 R$    216.44     2023-12-04


In [ ]:
perf_vendedor = (
    df[df["vendedor"] != "Não identificado"]
    .groupby("vendedor")
    .agg(
        n_vendas=("id_venda", "count"),
        faturamento=("valor_total", "sum"),
        ticket_medio=("valor_total", "mean")
    )
    .sort_values("faturamento", ascending=False)
    .reset_index()
)
perf_vendedor["participacao_pct"] = (
    perf_vendedor["faturamento"] / perf_vendedor["faturamento"].sum() * 100
)

print("Performance por vendedor:\n")
print(f"  {'Vendedor':<18} {'Vendas':>7} {'Faturamento':>13} {'Part.':>7} {'Ticket médio':>13}")
print(f"  {'-'*18} {'-'*7} {'-'*13} {'-'*7} {'-'*13}")
for _, row in perf_vendedor.iterrows():
    print(
        f"  {row['vendedor']:<18} "
        f"{row['n_vendas']:>7} "
        f"R$ {row['faturamento']:>9,.2f} "
        f"{row['participacao_pct']:>6.1f}% "
        f"R$ {row['ticket_medio']:>9,.2f}"
    )

Performance por vendedor:

  Vendedor            Vendas   Faturamento   Part.  Ticket médio
  ------------------ ------- ------------- ------- -------------
  Carlos Souza            96 R$ 16,048.78   25.6% R$    167.17
  Fernanda Costa         108 R$ 15,813.59   25.2% R$    146.42
  Roberto Dias           102 R$ 15,469.26   24.7% R$    151.66
  Ana Lima                97 R$ 15,416.20   24.6% R$    158.93


In [ ]:
ordem_dias = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
nomes_dias = {
    "Monday": "Segunda", "Tuesday": "Terça", "Wednesday": "Quarta",
    "Thursday": "Quinta", "Friday": "Sexta", "Saturday": "Sábado", "Sunday": "Domingo"
}

vendas_dia = (
    df.groupby("dia_semana")["valor_total"]
    .sum()
    .reindex(ordem_dias, fill_value=0)
    .reset_index()
)
vendas_dia.columns = ["dia_semana", "faturamento"]
vendas_dia["dia_pt"] = vendas_dia["dia_semana"].map(nomes_dias)

print("Faturamento por dia da semana:\n")
for _, row in vendas_dia.iterrows():
    barra = "█" * int(row["faturamento"] / vendas_dia["faturamento"].max() * 25)
    print(f"  {row['dia_pt']:<10}  {barra:<25}  R$ {row['faturamento']:>8,.2f}")

Faturamento por dia da semana:

  Segunda     ███████████████████████    R$ 13,052.02
  Terça       ████████████████████████   R$ 13,482.33
  Quarta      █████████████████          R$ 9,846.84
  Quinta      ██████████████             R$ 7,826.51
  Sexta       █████████████████          R$ 9,771.35
  Sábado      █████████████████████████  R$ 13,964.92
  Domingo     █████████████████          R$ 9,983.22


In [ ]:
import os
os.makedirs("data/indicators", exist_ok=True)

# Salvar cada tabela para usar nos gráficos
fat_mensal.to_csv("fat_mensal.csv", index=False, encoding="utf-8-sig")
top_produtos.to_csv("top_produtos.csv", index=False, encoding="utf-8-sig")
ticket_pagamento.to_csv("ticket_pagamento.csv", index=False, encoding="utf-8-sig")
top_clientes.to_csv("top_clientes.csv", index=False, encoding="utf-8-sig")
perf_vendedor.to_csv("perf_vendedor.csv", index=False, encoding="utf-8-sig")
vendas_dia.to_csv("vendas_dia.csv", index=False, encoding="utf-8-sig")

print("Arquivos de indicadores salvos:")
print("  fat_mensal.csv")
print("  top_produtos.csv")
print("  ticket_pagamento.csv")
print("  top_clientes.csv")
print("  perf_vendedor.csv")
print("  vendas_dia.csv")

Arquivos de indicadores salvos:
  fat_mensal.csv
  top_produtos.csv
  ticket_pagamento.csv
  top_clientes.csv
  perf_vendedor.csv
  vendas_dia.csv
